# Text Classification with the HuggingFace Ecosystem
## Author's note

This written by Wee Zen and quote "was entirely generated with ChatGPT 👻"...

We will explore training a model from scratch first, but using modernized building blocks such as HF tokenizers and datasets. Then, we will use pre-trained models, such as Glove (if I can get it to work) and BERT.

## Imports and setup

In [2]:
!pip install -q datasets tokenizers 


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## The magic of Datasets
The team at HF have made it really easy to load data (and actually everything else) - pretty cool! Let's see how easy it is.

In [2]:
from datasets import load_dataset
train_ds = load_dataset("SetFit/ag_news", split="train")
test_ds = load_dataset("SetFit/ag_news", split="test")

As you can see, all you need is to memorize `load_dataset`, and the URI of the dataset you want. Neat!  

Now, convert it into a Pytorch-sane dataset.

In [3]:
train_ds.set_format(type="torch")
test_ds.set_format(type="torch")

## Tokenizers are too powerful
Now this is something you shouldn't use unless you know you are very well versed already at implementing tokenizers and using vocabularies. If you get too attached and dependent on this library you will never recover (like smoking D:)

In [4]:
from tokenizers import Tokenizer
from tokenizers.models import BPE # In this case, we use BPE as it is super common

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

That's all the code you need for a tokenizer. No `class`, no `__init__`, no pain.  
It's still trainable though! (as in it's not pretrained by default) so we have to poupulate it. This means memorizing more code :(

In [5]:
VOCAB_SIZE = 8192

from tokenizers.trainers import BpeTrainer
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=8192)

The `[PAD]` token is for padding, which we can easily enable using the following feature:

In [6]:
tokenizer.enable_padding(pad_token="[PAD]")

There are also pre-tokenizers for the tokenizer to further control it. So for example if you don't want your tokens to be larger than one word, you can use the `Whitespace` preprocessor.

In [7]:
from tokenizers.pre_tokenizers import Whitespace
tokenizer.pre_tokenizer = Whitespace()

Lastly, there are normalizers which are used to remove things that we don't want, like accents on characters and uppercase letters.

In [8]:
from tokenizers.normalizers import Lowercase, StripAccents, Sequence
tokenizer.normalizer = Sequence([
    Lowercase(), StripAccents()
])

## Combining the two powers
"Never, in any circumstance, use the Tokenizers with the Datasets. It will not lead you to a place with loads of treasure, like efficiency and brain rot." ~ Steve from the Minecraft movie, probably.

Using the tokenizer itself is quite powerful. But we first need to train it on our dataset.  

To answer that, we need to talk about ~~parallel universes~~ the datasets library. Observe the voodoo brainrot P1-level code below.

In [9]:
train_ds

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 120000
})

In [10]:
text_only_train_ds = train_ds.select_columns("text") # This gives only the text column
text_only_train_ds

Dataset({
    features: ['text'],
    num_rows: 120000
})

In [11]:
def batch_iterator(ds, batch_size=10000):
    for batch in ds.iter(batch_size):
        yield batch["text"] # Following the name of the column, like indexing

In [12]:
next(batch_iterator(text_only_train_ds, 5)) # for example, only get 5

["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
 "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.",
 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.',
 'Oil prices soar to all-time record, posing new menace to US economy (A

We now have an iterator. Hooray! Why? Because the tokenizer library was made to work with datasets so large that anything that requires it to be all loaded into memory is a no-go. Thus, the only two methods supported are training from a file `train()` and training from an iterator `train_from_iterator()` - which we've just created.

In [13]:
tokenizer.train_from_iterator(
    batch_iterator(text_only_train_ds, 10000), # 10000 is sane as its just text,
    trainer # the one we defined just now!!
)

That barely took 5 seconds. Now we can tokenize anything!

In [14]:
encoding = tokenizer.encode("[Verse]La-la-la-lava, ch-ch-ch-chicken\nSteve's Lava Chicken, yeah, it's tasty as hell\nOoh, mamacita, now you're ringing the bell")
encoding

Encoding(num_tokens=58, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [15]:
print("Tokens:", encoding.tokens)
print()
print("IDs:", encoding.ids)

Tokens: ['[UNK]', 'ver', 'se', '[UNK]', 'la', '-', 'la', '-', 'la', '-', 'la', 'va', ',', 'ch', '-', 'ch', '-', 'ch', '-', 'ch', 'icken', 'steve', "'", 's', 'la', 'va', 'ch', 'icken', ',', 'y', 'e', 'ah', ',', 'it', "'", 's', 't', 'ast', 'y', 'as', 'hell', 'o', 'oh', ',', 'm', 'am', 'ac', 'it', 'a', ',', 'now', 'you', "'", 're', 'ring', 'ing', 'the', 'bell']

IDs: [0, 118, 87, 0, 257, 12, 257, 12, 257, 12, 257, 1495, 11, 86, 12, 86, 12, 86, 12, 86, 5904, 2875, 7, 49, 257, 1495, 86, 5904, 11, 55, 35, 1154, 11, 74, 7, 49, 50, 151, 55, 76, 6604, 45, 2752, 11, 43, 95, 94, 74, 31, 11, 623, 769, 7, 62, 2048, 73, 64, 2152]


For those of us who remember calling the numbers "tokens" - well, they're called "IDs" here so get used to the new world order.  
Lastly, let's tokenize our datasets and create dataloaders.

In [17]:
def encode_fn(example):
    return {"ids": [ x.ids for x in tokenizer.encode_batch(example["text"])]}
train_ds_tokenized = train_ds.shuffle().map(encode_fn, batched=True, batch_size=32)
test_ds_tokenized = test_ds.shuffle().map(encode_fn, batched=True, batch_size=32)
next(iter(train_ds_tokenized))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

{'text': 'LA hotel locks out laundry union, strike feared A major Los Angeles hotel locked out laundry workers Thursday in a move the general manager said could lead to a strike by union employees that, in turn, threatens to spur action in two other major US cities.',
 'label': tensor(2),
 'label_text': 'Business',
 'ids': tensor([ 257, 3506, 3077,  161,  180,  257,  259,  317, 1086,   11, 1576, 5655,
           31,  827,  581, 1839, 3506, 6046,  180,  257,  259,  317, 1428,  379,
           57,   31,  986,   64, 1151, 1646,  178,  532,  298,   72,   31, 1576,
          163, 1086, 2660,  143,   11,   57, 1108,   11, 4881,   72, 3931, 1565,
           57,  281,  479,  827,  103, 4247,   13,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0])}

In [18]:
train_dataloader = torch.utils.data.DataLoader(train_ds_tokenized, batch_size=32, num_workers=3, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(test_ds_tokenized, batch_size=32, num_workers=3, shuffle=False)
next(iter(train_dataloader))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{'text': ['LA hotel locks out laundry union, strike feared A major Los Angeles hotel locked out laundry workers Thursday in a move the general manager said could lead to a strike by union employees that, in turn, threatens to spur action in two other major US cities.',
  "U.S. Image Slides, but Americans Popular (AP) AP - America's popularity around the world has taken a beating in recent years, according to a set of coordinated polls conducted in 10 different countries. But the survey also found that despite widespread animosity toward President Bush, huge majorities said they have a good opinion of Americans.",
  'Indiana, Ole Miss, BYU Dismiss Coaches (AP) AP - Gerry DiNardo, Gary Crowton and David Cutcliffe all became unemployed coaches Wednesday, upping the total coaching vacancies in Division I-A to 15.',
  'Microsoft lifts the lid on SP2 SOFTWARE FIRM Microsoft has revealed how many fixes and updates are under the bonnet of its megapatch Windows XP Service Pack 2. ',
  "Yankees'

## Building the simple model
In such a short time we can already build the model. How cool is that! 

In [42]:
!pip install -q torchinfo

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from torchinfo import summary

In [20]:
class AGNewsLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, out_dim, layers=1, bidirectional=False, dropout=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=layers, batch_first=True, bidirectional=bidirectional, dropout=dropout)
        self.fc = nn.Linear((int(bidirectional)+1)*hidden_dim, out_dim) # Funny bidirectional code doubles the hidden dim if bidirectional!
    def forward(self, x):
        x = self.embedding(x)
        out, (hidden_outs, cell_outs) = self.lstm(x)
        if self.bidirectional:
            x = torch.cat((hidden_outs[-1], hidden_outs[-2]), dim=1)
        else:
            x = hidden_outs[-1]
        x = self.fc(x) # -1 of the 0th dimension, which is the layer. For multi-level LSTMs, we get the highest level, which should be the most abstract.
        return x

# Initialize a default model to check the parameters
# BUG: Bidirectional doesn't work 
model = AGNewsLSTM(VOCAB_SIZE, 100, 64, 4)
summary(model, input_data=torch.zeros((32, 128), dtype=torch.int64)) # Batch Size x Input Lengths

Layer (type:depth-idx)                   Output Shape              Param #
AGNewsLSTM                               [32, 4]                   --
├─Embedding: 1-1                         [32, 128, 100]            819,200
├─LSTM: 1-2                              [32, 128, 64]             42,496
├─Linear: 1-3                            [32, 4]                   260
Total params: 861,956
Trainable params: 861,956
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 200.29
Input size (MB): 0.03
Forward/backward pass size (MB): 5.37
Params size (MB): 3.45
Estimated Total Size (MB): 8.86

## Training the model

In [21]:
from tqdm import tqdm

In [23]:
def accuracy_fn(y_pred, y_true):
    return (torch.argmax(y_pred, dim=1)==y_true).sum().item() / y_true.shape[0]

In [24]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for b, data in enumerate(tqdm(train_dataloader)):
        X, y = data["ids"], data["label"]
        y_logits = model(X)
        y_pred = torch.softmax(y_logits, dim=1)
        loss = loss_fn(y_pred, y)
        epoch_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    epoch_loss /= len(train_dataloader)
    print(f"Train loss: {epoch_loss}")
    
    model.eval()
    with torch.inference_mode():
        eval_loss = 0
        eval_acc = 0
        for b, data in enumerate(tqdm(test_dataloader)):
            X, y = data["ids"], data["label"]
            y_logits = model(X)
            y_pred = y_logits
            loss = loss_fn(y_pred, y)
            eval_loss += loss.item()
            eval_acc += accuracy_fn(y_pred, y)
        eval_loss /= len(test_dataloader)
        eval_acc /= len(test_dataloader)
    
    print(f"Eval loss: {eval_loss}, Eval acc: {eval_acc}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...                                           | 0/3750 [00:00<?, ?it/s]
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variabl

Train loss: 1.3541897123654683


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...                                            | 0/238 [00:00<?, ?it/s]
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variabl

Eval loss: 1.7308160211859631, Eval acc: 0.4321165966386555


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...                                           | 0/3750 [00:00<?, ?it/s]
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variabl

Train loss: 1.2263738732179006


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...                                            | 0/238 [00:00<?, ?it/s]
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variabl

Eval loss: 1.4281722898242855, Eval acc: 0.5990021008403361


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...                                           | 0/3750 [00:00<?, ?it/s]
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variabl

KeyboardInterrupt: 

### Training a big boy model

In [31]:
model = AGNewsLSTM(VOCAB_SIZE, 300, 512, 4, layers=3, bidirectional=True, dropout=0.2)
summary(model, input_data=torch.zeros((32, 128), dtype=torch.int64)) # Batch Size x Input Lengths

Layer (type:depth-idx)                   Output Shape              Param #
AGNewsLSTM                               [32, 4]                   --
├─Embedding: 1-1                         [32, 128, 300]            2,457,600
├─LSTM: 1-2                              [32, 128, 1024]           15,933,440
├─Linear: 1-3                            [32, 4]                   4,100
Total params: 18,395,140
Trainable params: 18,395,140
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 65.34
Input size (MB): 0.03
Forward/backward pass size (MB): 43.39
Params size (MB): 73.58
Estimated Total Size (MB): 117.00

In [32]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

Wow... it took 3 hours to train just one epoch. It's good accuracy - but we can definitely do better if we didn't have to train for so long.  

That's why we can use pretrained models!

## Using Pre-trained models

I'm going to use ~~DistilBERT~~ ~~MobileBERT~~ ~~ALBERT~~ because BERT is cool but I'm on CPU and my skin and bones will wilt away before it finishes fine tuning. Somehow, DistilBERT is also too slow. MobileBERT is also only marginally faster than DistilBERT. ~~ALBERT is also marginally faster but I'm too lazy to swap it back.~~ My code was wrong, DistilBERT is still the fastest...  
Using pretrained models uses the other, more well known `transformers` library by HF. Let's see how to load it!

In [3]:
!pip install -q transformers


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [73]:
from transformers import DistilBertTokenizerFast, DistilBertModel

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
bert_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

bert_model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [74]:
sample_text = "[Verse]La-la-la-lava, ch-ch-ch-chicken\nSteve's Lava Chicken, yeah, it's tasty as hell\nOoh, mamacita, now you're ringing the bell"
encoded_sample = tokenizer.encode(sample_text, return_tensors="pt")
bert_model(encoded_sample).last_hidden_state[:, 0, :].shape

torch.Size([1, 768])

In [ ]:
class AGNewsPretrainedBERT(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.bert_model = DistilBertModel.from_pretrained("distilbert-base-uncased")
        for param in self.bert_model.parameters():
            param.requires_grad = False
        self.fc = nn.Linear(768, out_dim)
    def forward(self, x, attention_mask=None):
        x = self.bert_model(x, attention_mask=attention_mask).last_hidden_state[:, 0, :]  # Get the weight of first token --> Acts like CLS token
        x = self.fc(x)
        return x

model = AGNewsPretrainedBERT(4).to(device)
summary(model, input_data=encoded_sample.to(device)

In [79]:
def encode_fn(example):
    encodings = tokenizer(example["text"], padding=True)
    return { "ids": encodings["input_ids"], "attention_mask": encodings["attention_mask"] }
bert_train_ds_tokenized = train_ds.shuffle().map(encode_fn, batched=True, batch_size=32)
bert_test_ds_tokenized = test_ds.shuffle().map(encode_fn, batched=True, batch_size=32)

train_dataloader = torch.utils.data.DataLoader(bert_train_ds_tokenized, batch_size=32, num_workers=2, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(bert_test_ds_tokenized, batch_size=32, num_workers=2, shuffle=False)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [65]:
next(iter(train_dataloader))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{'text': ['USC Thrives in Adversity, Tops Stanford (AP) AP - Though Southern California went to halftime down 11 points, Matt Grootegoed knew the Trojans were going to win as soon as he stepped into the locker room.',
  'Asian Stocks Gain, Led by Aluminum Corp.; BHP, Rio Tinto Climb Asian stocks rose after comments by Vice Premier Huang Ju increased optimism that China won #39;t adopt new measures to slow economic growth.',
  'Two Investment Banks Settle with SEC (Reuters) Reuters - Deutsche Bank Securities Inc.  and\\Thomas Weisel Partners agreed to pay a combined  #36;100 million to\\settle charges involving conflicts of interest between research\\and investment banking, U.S. regulators said on Thursday.',
  'Roma Form Reappears At Siena Francesco Totti scored a capping brace as crisis-stricken AS Roma got well with a 4-0 away rout of Siena. The Giallorossi built a quick 2-0 lead with Serie A top scorer Vincenzo Montella poaching a pair within 10 second half minutes.',
  "Harmony Iss

In [80]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5) # Smaller LR!

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for b, data in enumerate(tqdm(train_dataloader)):
        X, y, attn_mask = data["ids"], data["label"], data["attention_mask"]
        y_logits = model(X, attn_mask)
        y_pred = y_logits
        loss = loss_fn(y_pred, y)
        epoch_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    epoch_loss /= len(train_dataloader)
    
    model.eval()
    with torch.inference_mode():
        eval_loss = 0
        for b, data in enumerate(tqdm(test_dataloader)):
            X, y = data["ids"], data["label"]
            y_logits = model(X)
            y_pred = torch.softmax(y_logits, dim=1)
            loss = loss_fn(y_pred, y)
            eval_loss += loss.item()
        eval_loss /= len(test_dataloader)

    print(f"Train loss: {epoch_loss}, Test loss: {eval_loss}")

On Colab, this nets me about ~86% accuracy in 2 epochs, with about 5 minutes per epoch. Not too shabby, but improvements to be made are:
- Gradually unfreeze BERT, about 3 epochs in
- Higher LR for linear but lower lr for BERT
- Train more but I need money...